In [1]:
import pandas as pd
import os
import numpy as np
import json

# ----------------------------------------------------
# File path
# ----------------------------------------------------
# NOTE: Please adjust these paths to your local environment
RAW_DATA_FOLDER = r"C:\Users\wb512463\OneDrive - WBG\Resilient Housing\Global analysis\data_raw\World Bank"
PROCESSED_DATA_FOLDER = r"C:\Users\wb512463\OneDrive - WBG\Resilient Housing\Global analysis\data_processed"

# 🚨 UPDATED: Using the new single file for both GDP and GDP per capita (PPP adjusted)
NEW_GDP_FILENAME = "63554440-3c2b-4267-a2b6-28763cb5728e_Data.csv"
FILE_GDP_PPP = os.path.join(RAW_DATA_FOLDER, NEW_GDP_FILENAME)
# Setting original variable names to the new file path for consistency in the execution block
FILE_GDP = FILE_GDP_PPP
FILE_GDP_PCAP = FILE_GDP_PPP 
# Standard World Bank files
FILE_POVERTY = os.path.join(RAW_DATA_FOLDER, "API_11_DS2_en_csv_v2_126266.csv")
FILE_POPULATION = os.path.join(RAW_DATA_FOLDER, "API_SP.POP.TOTL_DS2_en_csv_v2_130083.csv")

FILE_WGI = os.path.join(RAW_DATA_FOLDER, "wgidataset_excel", "wgidataset.xlsx")
FILE_DB_CONST = os.path.join(RAW_DATA_FOLDER, "Dealing with Construction Permits.xlsx")

# 🚨 NEW: Regionデータのファイルパス定義
FILE_CLASS = os.path.join(RAW_DATA_FOLDER, "CLASS.xlsx")


# ----------------------------------------------------
# Country list
# ----------------------------------------------------
COUNTRY_LIST = [
    'Aruba', 'Afghanistan', 'Angola', 'Albania', 'Andorra', 'United Arab Emirates', 'Argentina',
    'Armenia', 'American Samoa', 'Antigua and Barbuda', 'Australia', 'Austria', 'Azerbaijan',
    'Burundi', 'Belgium', 'Benin', 'Burkina Faso', 'Bangladesh', 'Bulgaria', 'Bahrain',
    'Bahamas, The', 'Bosnia and Herzegovina', 'Belarus', 'Belize', 'Bermuda', 'Bolivia',
    'Brazil', 'Barbados', 'Brunei Darussalam', 'Bhutan', 'Botswana', 'Central African Republic',
    'Canada', 'Switzerland', 'Channel Islands', 'Chile', 'Chad','China', "Cote d'Ivoire", 'Cameroon',
    'Congo, Dem. Rep.', 'Congo, Rep.', 'Colombia', 'Comoros', 'Cabo Verde', 'Costa Rica',
    'Cuba', 'Curacao', 'Cayman Islands', 'Cyprus', 'Czechia', 'Germany', 'Djibouti', 'Dominica',
    'Denmark', 'Dominican Republic', 'Algeria', 'Ecuador', 'Egypt, Arab Rep.', 'Eritrea', 'Spain',
    'Estonia', 'Ethiopia', 'Finland', 'Fiji', 'France', 'Faroe Islands', 'Micronesia, Fed. Sts.',
    'Gabon', 'United Kingdom', 'Georgia', 'Ghana', 'Gibraltar', 'Guinea', 'Gambia, The',
    'Guinea-Bissau', 'Equatorial Guinea', 'Greece', 'Grenada', 'Greenland', 'Guatemala',
    'Guam', 'Guyana', 'Hong Kong SAR, China', 'Honduras', 'Croatia', 'Haiti', 'Hungary',
    'Indonesia', 'Isle of Man', 'India', 'Ireland', 'Iran, Islamic Rep.', 'Iraq', 'Iceland',
    'Israel', 'Italy', 'Jamaica', 'Jordan', 'Japan', 'Kazakhstan', 'Kenya', 'Kyrgyz Republic',
    'Cambodia', 'Kiribati', 'St. Kitts and Nevis', 'Korea, Rep.', 'Kuwait', 'Lao PDR', 'Lebanon',
    'Liberia', 'Libya', 'St. Lucia', 'Liechtenstein', 'Sri Lanka', 'Lesotho', 'Lithuania',
    'Luxembourg', 'Latvia', 'Macao SAR, China', 'St. Martin (French part)', 'Morocco', 'Monaco',
    'Moldova', 'Madagascar', 'Maldives', 'Mexico', 'Marshall Islands', 'North Macedonia', 'Mali',
    'Malta', 'Myanmar', 'Montenegro', 'Mongolia', 'Northern Mariana Islands', 'Mozambique',
    'Mauritania', 'Mauritius', 'Malawi', 'Malaysia', 'Namibia', 'New Caledonia', 'Niger', 'Nigeria',
    'Nicaragua', 'Netherlands', 'Norway', 'Nepal', 'Nauru', 'New Zealand', 'Oman', 'Pakistan',
    'Panama', 'Peru', 'Philippines', 'Palau', 'Papua New Guinea', 'Poland', 'Puerto Rico (US)',
    'Korea, Dem. People\'s Rep.', 'Portugal', 'Paraguay', 'West Bank and Gaza', 'French Polynesia',
    'Qatar', 'Romania', 'Russian Federation', 'Rwanda', 'Saudi Arabia', 'Sudan', 'Senegal',
    'Singapore', 'Solomon Islands', 'Sierra Leone', 'El Salvador', 'San Marino', 'Somalia, Fed. Rep.',
    'Serbia', 'South Sudan', 'Sao Tome and Principe', 'Suriname', 'Slovak Republic', 'Slovenia',
    'Sweden', 'Eswatini', 'Sint Maarten (Dutch part)', 'Seychelles', 'Syrian Arab Republic',
    'Turks and Caicos Islands', 'Togo', 'Thailand', 'Tajikistan', 'Turkmenistan', 'Timor-Leste',
    'Tonga', 'Trinidad and Tobago', 'Tunisia', 'Turkiye', 'Tuvalu', 'Tanzania', 'Uganda', 'Ukraine',
    'Uruguay', 'United States', 'Uzbekistan', 'St. Vincent and the Grenadines', 'Venezuela, RB',
    'British Virgin Islands', 'Virgin Islands (U.S.)', 'Viet Nam', 'Vanuatu', 'Samoa', 'Kosovo',
    'Yemen, Rep.', 'South Africa', 'Zambia', 'Zimbabwe',
]

# ----------------------------------------------------
# 📌 Existing Data Processing Function - ✅ Modified
# ----------------------------------------------------

def load_and_clean_wb_data(file_path: str, country_list: list) -> pd.DataFrame:
    """
    Loads World Bank time series data, filters by country, extracts the latest value,
    and returns it in a wide format using the indicator code as column names.

    🚨 Modification Points:
    1. Added logic to dynamically set `skiprows` based on the file name to handle
       the new GDP file format (skiprows=0) and old formats (skiprows=4).
    2. Adjusted column identification for different file header types.
    """
    # 🚨 Dynamic skiprows setting
    if os.path.basename(file_path) == NEW_GDP_FILENAME:
        # New GDP file: Header is the first row (skiprows=0)
        skip_rows = 0
    else:
        # Other files (Poverty, Population): Header is the 5th row (skiprows=4)
        skip_rows = 4

    # Load data
    df = pd.read_csv(file_path, skiprows=skip_rows)
    
    # Data Shaping (Wide to Long)
    # Series Name / Indicator Name change by file, but represent the same logical column
    if skip_rows == 0:
         # New GDP file uses 'Series Name' and 'Series Code'
        id_vars = ['Country Name', 'Country Code', 'Series Name', 'Series Code']
    else:
        # Old World Bank files use 'Indicator Name' and 'Indicator Code'
        id_vars = ['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code']
    
    # 🚨 Year Column Identification and Cleaning
    # Extract the year from column names (e.g., '2024 [YR2024]' or '2024') and select only valid years
    
    # Columns for the new file format ('2024 [YR2024]')
    year_cols_raw_new = [col for col in df.columns if isinstance(col, str) and '[' in col and 'YR' in col]
    # Columns for the old file format ('2024')
    year_cols_raw_old = [col for col in df.columns if isinstance(col, str) and col.isdigit()]
    
    # Function to extract the year
    def extract_year(col_name, is_new_format):
        try:
            if is_new_format:
                # Extract '2024' from '2024 [YR2024]'
                year_str = str(col_name).split(' ')[0]
            else:
                # If '2024', keep as is
                year_str = str(col_name)
                
            year = int(year_str)
            if 1960 <= year <= 2025: # Year range check
                return year
            return None
        except:
            return None
            
    # Parse years for the new file format
    year_map_new = {col: extract_year(col, True) for col in year_cols_raw_new}
    year_map_clean = {k: str(v) for k, v in year_map_new.items() if v is not None}
    
    # Parse years for the old file format (avoiding overlap with new format)
    year_map_old = {col: extract_year(col, False) for col in year_cols_raw_old if col not in year_map_clean.keys()}
    year_map_clean.update({k: str(v) for k, v in year_map_old.items() if v is not None})
    
    # Use only parsed years as data columns
    df.rename(columns=year_map_clean, inplace=True)
    year_cols = list(year_map_clean.values())
    
    # Rename columns to unify Series Code / Indicator Code (only if exists)
    if 'Series Code' in df.columns:
        df.rename(columns={'Series Code': 'Indicator Code'}, inplace=True)
    if 'Series Name' in df.columns:
        df.rename(columns={'Series Name': 'Indicator Name'}, inplace=True)
    
    # Convert to long format
    df_long = df.melt(id_vars=['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code'], 
                        value_vars=year_cols, var_name='Year', value_name='Value')

    # Type conversion
    df_long['Year'] = pd.to_numeric(df_long['Year'], errors='coerce').astype('Int64')
    df_long['Value'] = pd.to_numeric(df_long['Value'], errors='coerce')

    # Country Name filtering and missing value removal
    df_filtered = df_long[df_long['Country Name'].isin(country_list)].copy()
    df_filtered.dropna(subset=['Value'], inplace=True)

    # Extract the latest value (sort and select the first row)
    df_filtered.sort_values(by=['Country Name', 'Indicator Code', 'Year'], ascending=[True, True, False], inplace=True)
    df_latest = df_filtered.groupby(['Country Name', 'Indicator Code']).first().reset_index()

    # ------------------------------------
    # 🔑 Pivot processing to avoid column name collisions
    # ------------------------------------

    # Create new column names from indicator code (Value and Year)
    df_latest['Indicator_Code_Clean'] = df_latest['Indicator Code'].str.replace('.', '_', regex=False)

    df_latest['Indicator_Latest_Value'] = df_latest['Indicator_Code_Clean'] + '_Value'
    df_latest['Indicator_Latest_Year'] = df_latest['Indicator_Code_Clean'] + '_Year'

    # Pivot for values
    df_value_pivot = df_latest.pivot_table(
        index=['Country Name', 'Country Code'],
        columns='Indicator_Latest_Value',
        values='Value'
    ).reset_index()

    # Pivot for years
    df_year_pivot = df_latest.pivot_table(
        index=['Country Name', 'Country Code'],
        columns='Indicator_Latest_Year',
        values='Year'
    ).reset_index()

    # Merge values and years
    df_final = pd.merge(df_value_pivot, df_year_pivot,
                         on=['Country Name', 'Country Code'],
                         how='outer')

    # ------------------------------------
    # Metadata Collection
    # ------------------------------------
    # Create a dictionary of indicator codes and original names for this file
    metadata = df_latest.set_index('Indicator_Code_Clean')['Indicator Name'].to_dict()

    print(f"✅ Processing Complete: {os.path.basename(file_path)}")

    return df_final, metadata

# ----------------------------------------------------
# 🚨 NEW: Data Processing Function 3: Region Classification (CLASS.xlsx)
# ----------------------------------------------------

def load_and_clean_region_data(file_path: str, country_list: list) -> pd.DataFrame:
    """
    Loads CLASS.xlsx data, extracts Country Name (Economy) and Region, 
    and returns a DataFrame containing only the filtered countries.
    """
    # Excelファイルを読み込みます。ヘッダーは1行目(index 0)です。
    df = pd.read_excel(file_path, header=0)
    
    # 必要な列を抽出: Economy (Country Name) と Region
    df_region = df[['Economy', 'Region']].copy()
    
    # 列名を統一します
    df_region.columns = ['Country Name', 'Region']
    
    # Country Nameでフィルタリングし、必要な国のみを抽出します
    df_final = df_region[df_region['Country Name'].isin(country_list)].copy()
    
    # 重複を削除します (Regionは国ごとにユニークなはずですが、念のため)
    df_final.drop_duplicates(subset=['Country Name'], inplace=True)

    print(f"✅ Processing Complete: {os.path.basename(file_path)} (Region)")
    
    # Country Codeが必要な場合に備え、空の列を追加しておきます (マージキーとして使用しない)
    df_final['Country Code'] = None 
    
    return df_final


# ----------------------------------------------------
# 🌟 Data Processing Function 1: Governance Index (WGI) - No Change
# ----------------------------------------------------

def load_and_clean_wgi(file_path: str, country_list: list) -> pd.DataFrame:
    """
    Loads WGI data, extracts the latest value, and returns it in a wide format.
    """
    # Read the Excel file, setting the header to the 1st row (index 0)
    df = pd.read_excel(file_path, header=0)
    
    # Extract only necessary columns (using the correct column names confirmed in the image)
    df = df[['countryname', 'year', 'indicator', 'estimate']].copy()
    
    # Column name cleanup and type conversion
    df.columns = ['Country Name', 'Year', 'Indicator', 'Value']
    
    # Verify values are numeric and convert the type
    df['Year'] = pd.to_numeric(df['Year'], errors='coerce').astype('Int64')
    df['Value'] = pd.to_numeric(df['Value'], errors='coerce')
    
    # Country Name filtering and missing value removal
    df_filtered = df[df['Country Name'].isin(country_list)].copy()
    df_filtered.dropna(subset=['Value'], inplace=True)
    
    # Extract the latest value (sort and select the first row)
    # WGI has indicator abbreviations (e.g., cc) in the Indicator column
    df_filtered.sort_values(by=['Country Name', 'Indicator', 'Year'], ascending=[True, True, False], inplace=True)
    df_latest = df_filtered.groupby(['Country Name', 'Indicator']).first().reset_index()
    
    # ------------------------------------
    # WGI column name creation and pivoting
    # ------------------------------------
    
    df_latest['Indicator_Latest_Value'] = 'WGI_' + df_latest['Indicator'] + '_Value'
    df_latest['Indicator_Latest_Year'] = 'WGI_' + df_latest['Indicator'] + '_Year'
    
    # Pivot for values
    df_value_pivot = df_latest.pivot_table(
        index='Country Name', 
        columns='Indicator_Latest_Value', 
        values='Value'
    ).reset_index()
    
    # Pivot for years
    df_year_pivot = df_latest.pivot_table(
        index='Country Name', 
        columns='Indicator_Latest_Year', 
        values='Year'
    ).reset_index()

    # Merge values and years
    df_final = pd.merge(df_value_pivot, df_year_pivot, 
                         on='Country Name', 
                         how='outer')
    
    print(f"✅ Processing Complete: {os.path.basename(file_path)} (WGI)")
    
    return df_final, {}

# ----------------------------------------------------
# 🌟 Data Processing Function 2: Doing Business (Construction) - No Change
# ----------------------------------------------------

def load_and_clean_db_const(file_path: str, country_list: list) -> pd.DataFrame:
    """
    Loads Doing Business Construction data and extracts the required scores.
    """
    # Read the entire Excel file without a header
    df = pd.read_excel(file_path, header=None)
    
    
    # Data for each country starts from the 11th row (index 10), so skip previous rows
    df_data = df.iloc[10:].copy() 
    
    # Extract user-specified columns by index: Column B(1), Column C(2), Column G(6)
    df_subset = df_data[[1, 2, 6]].copy()
    
    # Column name cleanup: Manually set column names based on index
    df_subset.columns = ['Country Name', 'DB_CONST_SCORE', 'DB_CONST_QUALITY_INDEX']
    
    # Type conversion
    df_subset['DB_CONST_SCORE'] = pd.to_numeric(df_subset['DB_CONST_SCORE'], errors='coerce')
    df_subset['DB_CONST_QUALITY_INDEX'] = pd.to_numeric(df_subset['DB_CONST_QUALITY_INDEX'], errors='coerce')

    # Country Name filtering
    df_final = df_subset[df_subset['Country Name'].isin(country_list)].copy()

    # --- 💡 NEW FIX: Set values outside the valid range (0-15) for DB_CONST_QUALITY_INDEX to NaN ---
    invalid_mask = (df_final['DB_CONST_QUALITY_INDEX'] > 15) | (df_final['DB_CONST_QUALITY_INDEX'] < 0)
    df_final.loc[invalid_mask, 'DB_CONST_QUALITY_INDEX'] = np.nan
    # ---------------------------------------------------------------------

    # --- 💡 NEW FIX: Also set DB_CONST_SCORE to NaN if it is 0 ---
    invalid_mask_score = (df_final['DB_CONST_SCORE'] == 0)
    df_final.loc[invalid_mask_score, 'DB_CONST_SCORE'] = np.nan
    # ----------------------------------------------------
    
    
    
    # Create Country Code column for merging
    df_final['Country Code'] = None 

    print(f"✅ Processing Complete: {os.path.basename(file_path)} (DB Construction)")
    
    return df_final, {}

# ----------------------------------------------------
# 📌 Data Processing and Integration Execution - ✅ MODIFIED
# ----------------------------------------------------

# 1. Process each file
print("--- Processing WB Time Series Data ---")

# 🚨 FIX: Load the combined GDP/GDP PCAP PPP file only once.
df_gdp_ppp_data, meta_gdp_ppp = load_and_clean_wb_data(FILE_GDP_PPP, COUNTRY_LIST)

# Poverty and Population
df_poverty, meta_poverty = load_and_clean_wb_data(FILE_POVERTY, COUNTRY_LIST)
df_population, meta_population = load_and_clean_wb_data(FILE_POPULATION, COUNTRY_LIST) 

print("\n--- Processing New Data ---")
df_wgi, meta_wgi = load_and_clean_wgi(FILE_WGI, COUNTRY_LIST)
df_db_const, meta_db_const = load_and_clean_db_const(FILE_DB_CONST, COUNTRY_LIST)

# 🚨 NEW: Regionデータを読み込みます
df_region = load_and_clean_region_data(FILE_CLASS, COUNTRY_LIST)


# 2. Integrating Processed Results (using Country Name and Country Code as keys)
# Start integration with the combined GDP/GDP_PCAP data
df_integrated = df_gdp_ppp_data.copy()

# Merge other time series data
df_integrated = pd.merge(df_integrated, df_poverty, on=['Country Name', 'Country Code'], how='outer')
df_integrated = pd.merge(df_integrated, df_population, on=['Country Name', 'Country Code'], how='outer')

# Merge WGI (as WGI does not have Country Code, use only Country Name as key)
df_integrated = pd.merge(df_integrated, df_wgi, on='Country Name', how='outer')

# Merge DB_Const (as DB_Const does not have Country Code, use only Country Name as key)
# Remove dummy Country Code from DB_Const
df_db_const.drop(columns=['Country Code'], inplace=True)
df_integrated = pd.merge(df_integrated, df_db_const, on='Country Name', how='outer')

# 🚨 NEW: Regionデータをマージします (Country Nameをキーとして使用)
# RegionデータにはCountry Codeは含まれていませんが、ここではCountry Nameでマージします
df_integrated = pd.merge(df_integrated, df_region[['Country Name', 'Region']], on='Country Name', how='left')


# Integrated Metadata
# 🚨 FIX: Only use meta_gdp_ppp once in the metadata dictionary.
FULL_METADATA = {**meta_gdp_ppp, **meta_poverty, **meta_population, **meta_wgi, **meta_db_const} 


# ----------------------------------------------------
# 📌 3. Column Name Filtering and Renaming
# ----------------------------------------------------

# Existing WB Indicators
# Use the new file's Indicator Code (GDP PPP 2021 adjusted, GDP per capita PPP adjusted)
INDICATOR_RENAME_MAP_RAW = {
    # UPDATED NAME: NY.GDP.MKTP.PP.KD (GDP, PPP, constant 2021 intl $)
    'NY_GDP_MKTP_PP_KD': 'GDP_PPP_Intl_2021',      
    # UPDATED NAME: NY.GDP.PCAP.PP.KD (GDP per capita, PPP, constant 2021 intl $)
    'NY_GDP_PCAP_PP_KD': 'GDP_PCAP_PPP_Intl_2021', 
    'SI_SPR_PCAP': 'Poverty_Survey_Mean_Income_PPP',
    'SI_POV_UMIC': 'Poverty_HC_Ratio_at_USD8_30',
    'SI_POV_NAHC': 'Poverty_HC_Ratio_National_Line',
    'SI_POV_LMIC': 'Poverty_HC_Ratio_at_USD4_20',
    'SI_POV_GINI': 'Gini_Index',
    'SI_POV_GAPS': 'Poverty_Gap_at_USD3_00_2021PPP',
    'SI.POV.LMIC.GP': 'Poverty_Gap_at_USD4_20_2021PPP',
    'SI.POV.UMIC.GP': 'Poverty_Gap_at_USD8_30_2021PPP',
    'SI.POV.GINI': 'Gini index',
    'EN_POP_SLUM_UR_ZS': 'Slum_Population_Urban_Pct',
    'SP_POP_TOTL': 'Population_Total',
    'SI.POV.DDAY' : 'Poverty_HC_Ratio_at_USD3_00',
}

# WGI Indicators (Mapping new column names to existing WGI column names)
WGI_RENAME_MAP = {
    'WGI_va_Value': 'WGI_Voice_Accountability_Value',
    'WGI_pv_Value': 'WGI_Political_Stability_Value',
    'WGI_ge_Value': 'WGI_Government_Effectiveness_Value',
    'WGI_rq_Value': 'WGI_Regulatory_Quality_Value',
    'WGI_rl_Value': 'WGI_Rule_of_Law_Value',
    'WGI_cc_Value': 'WGI_Control_of_Corruption_Value',
    'WGI_va_Year': 'WGI_Voice_Accountability_Year',
    'WGI_pv_Year': 'WGI_Political_Stability_Year',
    'WGI_ge_Year': 'WGI_Government_Effectiveness_Year',
    'WGI_rq_Year': 'WGI_Regulatory_Quality_Year',
    'WGI_rl_Year': 'WGI_Rule_of_Law_Year',
    'WGI_cc_Year': 'WGI_Control_of_Corruption_Year',
}

# Doing Business Indicators (Direct Mapping)
DB_RENAME_MAP = {
    'DB_CONST_SCORE': 'DB_Construction_Permits_Score',
    'DB_CONST_QUALITY_INDEX': 'DB_Building_Quality_Index',
}

# Create the final rename map with Value/Year suffixes
FINAL_RENAME_MAP = {}
# 🚨 NEW: Regionを追加
required_raw_cols = ['Country Name', 'Country Code', 'Region'] # 'Country Name', 'Country Code', 'Region' are mandatory

# Existing WB Indicators
for raw_code, simple_name in INDICATOR_RENAME_MAP_RAW.items():
    # Indicator codes use underscores in the DataFrame
    value_col = f"{raw_code.replace('.', '_')}_Value"
    year_col = f"{raw_code.replace('.', '_')}_Year"
    
    FINAL_RENAME_MAP[value_col] = f"WB_{simple_name}_Value"
    FINAL_RENAME_MAP[year_col] = f"WB_{simple_name}_Year"
    
    required_raw_cols.extend([value_col, year_col])

# WGI Indicators
for raw_col, final_col in WGI_RENAME_MAP.items():
    FINAL_RENAME_MAP[raw_col] = f"WB_{final_col}"
    required_raw_cols.append(raw_col)

# DB Indicators
for raw_col, final_col in DB_RENAME_MAP.items():
    FINAL_RENAME_MAP[raw_col] = f"WB_{final_col}"
    required_raw_cols.append(raw_col)


# Filter for necessary columns (Country Code might be NaN for some rows)
df_filtered = df_integrated[[col for col in required_raw_cols if col in df_integrated.columns]].copy()

# Rename columns
df_final = df_filtered.rename(columns=FINAL_RENAME_MAP)

# Set Country Name as index (consistency with the initial code)
df_final = df_final.rename(columns={'Country Name': 'CountryName'}).set_index('CountryName')

# ----------------------------------------------------
# 📌 4. Column Order Arrangement (Value -> Year order) - ✅ MODIFIED
# ----------------------------------------------------

# Order Logic: Country Code -> 🚨 Region -> WB Value/Year Pairs -> WGI Value/Year Pairs -> DB Scores
# 🚨 NEW: RegionをCountry Codeのすぐ後に追加
new_column_order = ['Country Code', 'Region']

# 1. WB Time Series Data (Value -> Year)
wb_bases = sorted(list(set([col.replace('_Value', '').replace('_Year', '') for col in FINAL_RENAME_MAP.values() if col.startswith('WB_') and ('WGI' not in col) and ('DB' not in col)])))
for base_name in wb_bases:
    value_col = f"{base_name}_Value"
    year_col = f"{base_name}_Year"
    if value_col in df_final.columns:
        new_column_order.append(value_col)
    if year_col in df_final.columns:
        new_column_order.append(year_col)

# 2. WGI Data (Value -> Year)
wgi_bases = ['WGI_Voice_Accountability', 'WGI_Political_Stability', 'WGI_Government_Effectiveness', 'WGI_Regulatory_Quality', 'WGI_Rule_of_Law', 'WGI_Control_of_Corruption']
for base_name in wgi_bases:
    value_col = f"WB_{base_name}_Value"
    year_col = f"WB_{base_name}_Year"
    if value_col in df_final.columns:
        new_column_order.append(value_col)
    if year_col in df_final.columns:
        new_column_order.append(year_col)

# 3. Doing Business Data (Score Only)
if 'WB_DB_Construction_Permits_Score' in df_final.columns:
    new_column_order.append('WB_DB_Construction_Permits_Score')
if 'WB_DB_Building_Quality_Index' in df_final.columns:
    new_column_order.append('WB_DB_Building_Quality_Index')

# Apply the final column order (skip non-existent columns)
df_final_ordered = df_final[[col for col in new_column_order if col in df_final.columns]]


# ----------------------------------------------------
# 📌 5. Export as Final CSV
# ----------------------------------------------------
export_file_name = "Country_WorldBank_Data_Cleaned_For_Merge.csv"
export_path = os.path.join(PROCESSED_DATA_FOLDER, export_file_name)

df_final_ordered.to_csv(export_path, encoding='utf-8-sig')

print("\n--- 📊 Integrated Dataset Overview (Final Check) ---")
print(f"Final Data Shape: {df_final_ordered.shape}")
print(f"Number of Columns: {len(df_final_ordered.columns)}")
print("--- First 5 Rows ---")
print(df_final_ordered.head())
print("--- Final Column Names and Order ---")
print(df_final_ordered.columns.tolist())

print("\n--- 🏁 Processing Complete ---")
print(f"🎉 Extended World Bank dataset successfully exported.")
print(f"Path: {export_path}")

--- Processing WB Time Series Data ---
✅ Processing Complete: 63554440-3c2b-4267-a2b6-28763cb5728e_Data.csv
✅ Processing Complete: API_11_DS2_en_csv_v2_126266.csv
✅ Processing Complete: API_SP.POP.TOTL_DS2_en_csv_v2_130083.csv

--- Processing New Data ---
✅ Processing Complete: wgidataset.xlsx (WGI)
✅ Processing Complete: Dealing with Construction Permits.xlsx (DB Construction)
✅ Processing Complete: CLASS.xlsx (Region)

--- 📊 Integrated Dataset Overview (Final Check) ---
Final Data Shape: (217, 44)
Number of Columns: 44
--- First 5 Rows ---
               Country Code                      Region  \
CountryName                                               
Afghanistan             AFG                  South Asia   
Albania                 ALB       Europe & Central Asia   
Algeria                 DZA  Middle East & North Africa   
American Samoa          ASM         East Asia & Pacific   
Andorra                 AND       Europe & Central Asia   

                WB_GDP_PCAP_PPP_Intl_2